# Custom CNN for Time Series Classification

This notebook demonstrates how to use a custom CNN model with PyTorch for time series classification.

In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Add the parent directory to the path to import from src
sys.path.append("..")

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

## 1. Load and Prepare Data

In [ ]:
def load_npy_dataset(file_path):
    """
    Load and prepare numpy dataset for time series classification.

    Args:
        file_path: Path to the .npy file containing the dataset

    Returns:
        Tuple containing (X_train, y_train, X_test, y_test)
    """
    data = np.load(file_path, allow_pickle=True).item()

    X_train = data["train"]["X"]
    y_train = np.array([int(x) for x in data["train"]["y"]])
    X_test = data["test"]["X"]
    y_test = np.array([int(x) for x in data["test"]["y"]])

    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"y_test shape: {y_test.shape}")

    return X_train, y_train, X_test, y_test

In [ ]:
# Specify the path to your dataset
dataset_path = "../data/raw/your_dataset.npy"  # Change this to your dataset path

# Load the dataset
X_train, y_train, X_test, y_test = load_npy_dataset(dataset_path)

In [ ]:
# Ensure data is in the right format for PyTorch (batch, channels, timesteps)
def prepare_data_for_cnn(X_train, X_test):
    # Check if data needs reshaping
    if len(X_train.shape) == 2:
        # For univariate time series: reshape from (batch, time) to (batch, 1, time)
        X_train = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
        X_test = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

    # If data has the shape (batch, time, channels), transpose to (batch, channels, time)
    elif len(X_train.shape) == 3 and X_train.shape[1] > X_train.shape[2]:
        X_train = np.transpose(X_train, (0, 2, 1))
        X_test = np.transpose(X_test, (0, 2, 1))

    return X_train, X_test


# Prepare data
X_train, X_test = prepare_data_for_cnn(X_train, X_test)
print(f"Prepared data shape - X_train: {X_train.shape}, X_test: {X_test.shape}")

## 2. Create PyTorch Dataset and DataLoader

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
# Split training data for validation
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# Create datasets
train_dataset = TimeSeriesDataset(X_train_split, y_train_split)
val_dataset = TimeSeriesDataset(X_val, y_val)
test_dataset = TimeSeriesDataset(X_test, y_test)

# Create data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## 3. Define the CNN Model

In [ ]:
class MultivariateTimeSeriesCNN(nn.Module):
    def __init__(self, num_channels, seq_length, num_classes):
        super(MultivariateTimeSeriesCNN, self).__init__()

        self.conv1 = nn.Conv1d(
            in_channels=num_channels, out_channels=64, kernel_size=3, padding=1
        )
        self.bn1 = nn.BatchNorm1d(64)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(
            in_channels=64, out_channels=128, kernel_size=3, padding=1
        )
        self.bn2 = nn.BatchNorm1d(128)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(2)

        self.flat_size = 128 * (seq_length // 4)

        self.fc1 = nn.Linear(self.flat_size, 256)
        self.fc_relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = x.view(x.size(0), -1)

        x = self.fc1(x)
        x = self.fc_relu(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x

In [ ]:
def count_parameters(model):
    """Count the number of trainable parameters in the model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# Set device (GPU if available, otherwise CPU)
device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

# Get model dimensions from data
num_channels = X_train.shape[1]
seq_length = X_train.shape[2]
num_classes = len(np.unique(y_train))

# Initialize model
model = MultivariateTimeSeriesCNN(num_channels, seq_length, num_classes).to(device)

# Print model summary
print(model)
print(f"Number of trainable parameters: {count_parameters(model):,}")

## 4. Train and Evaluate the Model

In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    num_epochs,
    device,
    patience=10,
):
    """Train the model with early stopping."""
    best_val_acc = 0
    epochs_without_improvement = 0
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "epoch_times": [],
    }

    for epoch in range(num_epochs):
        epoch_start_time = time.time()

        # Training phase
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += batch_y.size(0)
            train_correct += (predicted == batch_y).sum().item()

        # Validation phase
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += batch_y.size(0)
                val_correct += (predicted == batch_y).sum().item()

        # Calculate metrics
        train_acc = 100 * train_correct / train_total
        val_acc = 100 * val_correct / val_total
        epoch_time = time.time() - epoch_start_time

        # Store history
        history["train_loss"].append(train_loss / len(train_loader))
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss / len(val_loader))
        history["val_acc"].append(val_acc)
        history["epoch_times"].append(epoch_time)

        # Print progress
        print(f"Epoch [{epoch + 1}/{num_epochs}] - Time: {epoch_time:.2f}s")
        print(
            f"Train Loss: {train_loss / len(train_loader):.4f}, Train Acc: {train_acc:.2f}%"
        )
        print(f"Val Loss: {val_loss / len(val_loader):.4f}, Val Acc: {val_acc:.2f}%")

        # Early stopping logic
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_without_improvement = 0
            torch.save(model.state_dict(), "cnn_best_model.pth")
            print(f"New best model saved with validation accuracy: {val_acc:.2f}%")
        else:
            epochs_without_improvement += 1
            print(
                f"Epochs without improvement: {epochs_without_improvement}/{patience}"
            )

            if epochs_without_improvement >= patience:
                print(f"Early stopping triggered after {epoch + 1} epochs")
                break

    # Load the best model
    model.load_state_dict(torch.load("cnn_best_model.pth"))
    return model, history

In [ ]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
learning_rate = 5e-5
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Train the model
num_epochs = 100
patience = 10

start_time = time.time()
model, history = train_model(
    model, train_loader, val_loader, criterion, optimizer, num_epochs, device, patience
)
fit_time = time.time() - start_time

print(f"\nTraining completed in {fit_time:.2f} seconds")

## 5. Plot Training History

In [ ]:
def plot_training_history(history):
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history["train_loss"], label="Train Loss")
    plt.plot(history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.title("Training and Validation Loss")
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(history["train_acc"], label="Train Accuracy")
    plt.plot(history["val_acc"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.legend()
    plt.title("Training and Validation Accuracy")
    plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_training_history(history)

## 6. Evaluate on Test Set

In [ ]:
def predict(model, data_loader, device):
    model.eval()
    predictions = []
    true_labels = []
    start_time = time.time()

    with torch.no_grad():
        for batch_X, batch_y in data_loader:
            batch_X = batch_X.to(device)
            outputs = model(batch_X)
            _, predicted = torch.max(outputs.data, 1)
            predictions.extend(predicted.cpu().numpy())
            true_labels.extend(batch_y.numpy())

    prediction_time = time.time() - start_time
    accuracy = accuracy_score(true_labels, predictions) * 100

    return predictions, true_labels, accuracy, prediction_time

In [ ]:
# Evaluate on training set
train_predictions, train_true, train_accuracy, train_pred_time = predict(
    model, DataLoader(train_dataset, batch_size=batch_size), device
)
print(f"Training Accuracy: {train_accuracy:.2f}%")
print(f"Training Prediction Time: {train_pred_time:.2f} seconds")
print(f"Training Samples/Second: {len(train_true) / train_pred_time:.2f}")

# Evaluate on test set
test_predictions, test_true, test_accuracy, test_pred_time = predict(
    model, test_loader, device
)
print(f"\nTest Accuracy: {test_accuracy:.2f}%")
print(f"Test Prediction Time: {test_pred_time:.2f} seconds")
print(f"Test Samples/Second: {len(test_true) / test_pred_time:.2f}")

In [ ]:
# Print classification report
print("Classification Report:")
print(classification_report(test_true, test_predictions))

## 7. Visualize Confusion Matrix

In [ ]:
import seaborn as sns

# Calculate confusion matrix
cm = confusion_matrix(test_true, test_predictions)
classes = np.unique(test_true)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

## 8. Save Model and Results

In [ ]:
import json
from datetime import datetime

# Create results directory if it doesn't exist
os.makedirs("../results", exist_ok=True)

# Save the model
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_save_path = f"../results/cnn_model_{timestamp}.pth"
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

# Calculate model size
model_size = (
    sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**2
)  # Size in MB

# Compile results
results = {
    "model_name": "CNN",
    "train_accuracy": float(train_accuracy),
    "test_accuracy": float(test_accuracy),
    "fit_time": float(fit_time),
    "pred_train_time": float(train_pred_time),
    "pred_test_time": float(test_pred_time),
    "train_samples_per_second": float(len(train_true) / train_pred_time),
    "test_samples_per_second": float(len(test_true) / test_pred_time),
    "model_size_mb": float(model_size),
    "num_parameters": int(count_parameters(model)),
    "dataset_name": os.path.basename(dataset_path).split(".")[0],
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

# Save results as JSON
results_save_path = f"../results/cnn_results_{timestamp}.json"
with open(results_save_path, "w") as f:
    json.dump(results, f, indent=4)
print(f"Results saved to {results_save_path}")

## 9. Analyze Predictions

In [ ]:
def visualize_predictions(
    X, true_labels, predicted_labels, indices=None, max_samples=5
):
    """Visualize time series with their true and predicted labels."""
    if indices is None:
        # Randomly select some samples
        indices = np.random.choice(
            range(len(true_labels)), min(max_samples, len(true_labels)), replace=False
        )

    n_samples = len(indices)
    plt.figure(figsize=(15, n_samples * 3))

    for i, idx in enumerate(indices):
        plt.subplot(n_samples, 1, i + 1)

        # Shape should be (channels, timesteps) for the selected example
        x = X[idx].cpu().numpy() if torch.is_tensor(X[idx]) else X[idx]

        if len(x.shape) == 2:  # Multivariate
            for j in range(x.shape[0]):  # Loop through channels
                plt.plot(x[j], label=f"Dim {j}")
            if x.shape[0] > 1:
                plt.legend(loc="upper right")
        else:  # Univariate
            plt.plot(x)

        correct = true_labels[idx] == predicted_labels[idx]
        color = "green" if correct else "red"
        title = f"True: {true_labels[idx]}, Predicted: {predicted_labels[idx]}"
        plt.title(title, color=color)
        plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
# Visualize correct predictions
correct_indices = np.where(np.array(test_true) == np.array(test_predictions))[0]
if len(correct_indices) > 0:
    print("Correctly Classified Examples:")
    selected_indices = np.random.choice(
        correct_indices, min(5, len(correct_indices)), replace=False
    )
    visualize_predictions(test_dataset.X, test_true, test_predictions, selected_indices)

In [ ]:
# Visualize incorrect predictions
incorrect_indices = np.where(np.array(test_true) != np.array(test_predictions))[0]
if len(incorrect_indices) > 0:
    print("Incorrectly Classified Examples:")
    selected_indices = np.random.choice(
        incorrect_indices, min(5, len(incorrect_indices)), replace=False
    )
    visualize_predictions(test_dataset.X, test_true, test_predictions, selected_indices)